# TSFresh-Features aus rohem, DPCA-, CVA- und ICA-gefiltertem TEP-Signal

Schwester-Notebook zu `TSFresh_PCA_DyCA.ipynb`: identische dreiphasige Pipeline
(Extraktion + Selektion auf Train, billige Test-Extraktion, LazyClassifier),
aber mit **DPCA**, **CVA** und **ICA** als Projektionen. `raw` läuft als
gemeinsamer Anker mit — dadurch sind die Ergebnisse beider Notebooks direkt
aneinander ausrichtbar.

## Die 16 Konfigurationen

| Konfiguration | Kanäle | Projektion |
|---|---|---|
| `raw` | 52 | keine — die 52 Prozessvariablen direkt |
| `dpca_4 … dpca_12` | 4, 6, 8, 10, 12 | DPCA pro Run: Scores der ersten *n* Hauptkomponenten der lag-erweiterten Matrix (L = 2, 156 Spalten) |
| `cva_4 … cva_12` | 4, 6, 8, 10, 12 | CVA pro Run: die ersten *n* kanonischen Variaten der Vergangenheit (p = f = 1) |
| `ica_4 … ica_12` | 4, 6, 8, 10, 12 | FastICA pro Run: *n* unabhängige Quellen, nach \|Exzess-Kurtosis\| absteigend geordnet |

Zusammen 52 + 40 + 40 + 40 = **172 Kanäle pro Run**. Die Parameter der
Verfahren (L, p/f, FastICA-Einstellungen) entsprechen den zugehörigen
Eigenwert-Notebooks `DPCA_Eigenwerte.ipynb` / `CVA_Eigenwerte.ipynb` /
`ICA_Eigenwerte.ipynb`.

## Kanallängen — kleine, dokumentierte Abweichung

Die Projektionen verkürzen die Zeitreihen methodisch bedingt unterschiedlich:
`raw`/`ica` behalten **480** Samples, `cva` hat **479** (der Zustand braucht
x_{t−1}), `dpca` hat **478** (L = 2 Lags). Das ist unkritisch, weil die
Regel „keine Längenartefakte" nur *innerhalb* einer Konfiguration greifen
muss: dort sind alle Runs und beide Splits exakt gleich lang. Zwischen
Konfigurationen werden Features nie verglichen — jede bekommt ihre eigene
Selektion und ihren eigenen Klassifizierer. `raw` bleibt bewusst bei 480,
damit die raw-Ergebnisse mit dem Schwester-Notebook bitidentisch sind und
der Cache geteilt werden kann (siehe unten).

## Gemeinsamer Cache mit `TSFresh_PCA_DyCA.ipynb`

`cache_dir` ist absichtlich **derselbe** (`tsfresh_cache` bzw.
`tsfresh_cache_smoke`): die `raw`-Konfiguration ist in beiden Notebooks
bitidentisch (gleiche Vorverarbeitung, gleiche Länge, gleiches Chunking) —
vorhandene raw-Chunks und die raw-Top-`top_k`-Auswahl werden dadurch
**wiederverwendet**, was bei bereits gelaufenem Schwester-Notebook rund
5,5 h spart. Zwei Konsequenzen:

- **`chunk_runs` muss auf 250 bleiben** (Wert des Schwester-Notebooks) —
  sonst würden dessen Cache-Dateien mit falscher Chunk-Aufteilung
  interpretiert und Runs stumm verloren gehen.
- Die Ergebnis-CSV heißt hier `tsfresh_summary_dpca_cva_ica.csv`, damit
  `tsfresh_summary.csv` des Schwester-Notebooks nicht überschrieben wird.

Die Prefixe `dpca_*`, `cva_*`, `ica_*` kollidieren mit nichts Bestehendem.

## Vorgehen (dreistufig)

**Phase A — Extraktion + Selektion auf TRAIN.** Je Konfiguration werden alle
TSFresh-Features auf allen Trainingsruns berechnet (`EfficientFCParameters`,
~780 Features je Kanal), daraus per Relevanztabelle die besten `top_k`
ausgewählt. Danach wird die volle Matrix sofort verworfen.

**Phase B — Testset.** `from_columns()` übersetzt die ausgewählten
Featurenamen zurück in `kind_to_fc_parameters`; auf dem Testset werden nur
diese Features berechnet. Die Selektion sieht das Testset nie.

**Phase C — LazyClassifier** je Konfiguration, dann Vergleich über alle 16.

## Laufzeit und Speicher — bitte vor dem Start lesen

Maßstab aus dem vollständigen Lauf des Schwester-Notebooks (8 Kerne,
`EfficientFCParameters`, alle 500 Runs je Fault): ~6,5 min TSFresh-Zeit pro
Kanal in Phase A. Hier: `raw` aus dem Cache ≈ 0 h (sonst ~5,5 h), die 15
projizierten Konfigurationen ≈ 120 Kanäle → Phase A grob **13 h**, dazu die
FastICA-Fits (~0,1–0,3 s je Run und ica-Konfiguration) mit **~2 h** in
Phase A und nochmals ~2 h in Phase B. Phase B insgesamt ~3–4 h, Phase C
~30 min. Zusammen Größenordnung **17–22 h** bei vorhandenem raw-Cache.

Speicher ist unkritischer als im Schwester-Notebook: die größte neue
Konfiguration hat 12 Kanäle (~9 400 Features/Run); nur `raw` (40 404
Features/Run) bleibt groß — und kommt in der Regel aus dem Cache. Die
Mechanik (float32-Matrizen, Konfigurationen strikt nacheinander,
Relevanztests in Spaltenblöcken, Chunk-Cache) ist identisch.

**Der Cache macht den Lauf unterbrechbar.** Jeder Chunk landet als Pickle in
`cache_dir`; ein Neustart überspringt alles bereits Berechnete.

> **`smoke_test` steht in der Konfigurationszelle.** Mit `True` läuft die
> komplette Pipeline in wenigen Minuten auf wenigen Runs durch. Für den
> echten Lauf auf `False` (am besten in tmux/über Nacht).

## Code-Struktur

Der gesamte Maschinenraum liegt im Paket [`tep/`](tep) und wird von allen
TEP-Notebooks geteilt. Dieses Notebook enthält nur noch, was es von
seinen Geschwistern unterscheidet: die Konfiguration, die Liste der
Projektions-Specs und die Aufrufe.

| Modul | Inhalt |
|---|---|
| `tep/core.py` | Spaltennamen, Splits, Cutoffs, Vorverarbeitung, lineare Algebra — geteilt mit `tep.eigen` |
| `tep/plotting.py` | Confusion-Matrix-Darstellung, ebenfalls geteilt |
| `tep/tsfresh/config.py` | `PipelineConfig` — alle Stellschrauben |
| `tep/tsfresh/projections.py` | Registry der Verfahren: `raw`, `pca`, `dyca`, `dpca`, `cva`, `ica`, `dycvda` |
| `tep/tsfresh/features.py` | Chunk-Cache-Extraktion, Feature-Ranking |
| `tep/tsfresh/pipeline.py` | Phase A / B / C |
| `tep/tsfresh/reporting.py` | Vergleichstabellen und Balkenplot |
| `tep/tsfresh/confusion.py` | Confusion-Matrizen und ihre drei Plots |

Eine Projektion ist ein `Projector` mit drei Angaben: wie sie heisst (der
Name ist zugleich Cache-Praefix), wie ihre Kanaele heissen und wie sie
rechnet. Ein eigenes Verfahren kommt über `tep.tsfresh.register(...)`
dazu, ohne dass hier etwas angefasst werden muss.

> Die Rechnungen sind zeilengetreu aus der früheren Notebook-Fassung
> uebernommen — Projektionen und Cache-Praefixe wurden vor dem Umbau
> ueber alle Konfigurationen und beide Skalierungsmodi als bitidentisch
> nachgewiesen. Der vorhandene Chunk-Cache bleibt damit gueltig.


In [ ]:
# ============================================================
# Imports - der gemeinsame Unterbau steckt im Paket tep
# ============================================================
# Wird tep/**.py bearbeitet, muss der Kernel neu gestartet werden;
# alternativ die beiden autoreload-Zeilen aktivieren.
# %load_ext autoreload
# %autoreload 2

# numpy/pandas werden hier nicht gebraucht, stehen aber fuer eigene
# Auswertungen am Ende des Notebooks bereit.
import numpy as np
import pandas as pd

from tep.tsfresh import (Pipeline, PipelineConfig, plot_confusion_detail,
                         plot_recall, versions)

print(versions())

In [ ]:
# ============================================================
# Konfiguration - die EINZIGE Stelle, an der geschraubt wird
# ============================================================
# Alle nicht gesetzten Felder stehen auf den Defaults aus
# tep/tsfresh/config.py (top_k=100, fc_mode='efficient',
# chunk_runs=250, run_length=480, lc_cv_folds=5, ...).
# smoke_test=True gibt einen winzigen Probelauf in einem
# eigenen Cache-Ordner - ueberschreibt also nichts.

DPCA_NS = [4, 6, 8, 10, 12]      # DPCA: behaltene Komponenten
CVA_NS = [4, 6, 8, 10, 12]       # CVA:  kanonische Variaten
ICA_NS = [4, 6, 8, 10, 12]       # ICA:  unabhaengige Quellen

CFG = PipelineConfig(
    configs=([("raw",)]
             + [("dpca", n) for n in DPCA_NS]
             + [("cva", n) for n in CVA_NS]
             + [("ica", n) for n in ICA_NS]),
    label="DPCA/CVA/ICA",
    summary_csv="tsfresh_summary_dpca_cva_ica.csv",
    cm_pred_csv="tsfresh_cm_predictions_dpca_cva_ica.csv",
    scaling_mode="scaler",

    # Verfahrensparameter. ACHTUNG: diese stecken NICHT im
    # Konfigurationsnamen und damit nicht im Cache-Praefix - bei
    # Aenderung vorher die betroffenen Chunks aus dem Cache loeschen.
    dpca_lags=2,                 # 52*(L+1) = 156 Spalten
    cva_past=1, cva_fut=1,       # Vergangenheits-/Zukunftsfenster
    cva_ridge_rel=1e-6,          # relative Ridge-Regularisierung
    ica_max_iter=1000, ica_tol=1e-3, ica_random_state=42,
)

## Rohdaten laden

Identisch zum Schwester-Notebook: die Runs werden **einmal** in ein Dictionary
`{(faultNumber, simulationRun): Array}` gelesen und danach für alle 16
Konfigurationen wiederverwendet — die TEP-CSVs (besonders
`TEP_Faulty_Testing.csv` mit 3,4 GB) sollen nur ein Mal durch den Parser.

Speicher: mit `float32` und nur den benötigten Spalten sind das je ~1,05 GB
für Train und Test (beide Splits werden auf `run_length` gekürzt). Train und
Test werden **nie gleichzeitig** gehalten — Phase A braucht nur Train,
Phase B nur Test.

Sortiert wird pro Run (nicht global), weil ein globales `sort_values` über
10 Mio. Zeilen eine komplette Kopie anlegen würde. DPCA (Lag-Stacking) und
CVA (Zeitpaare) brauchen die zeitliche Ordnung zwingend; für ICA hält sie die
FastICA-Schätzung reproduzierbar.

### `uniform_length` und `run_length`

Beide Schalter übernehmen unverändert die Begründung aus
`TSFresh_PCA_DyCA.ipynb`: längenabhängige TSFresh-Features (`length`,
`abs_energy`, `sum_values`, `count_above_mean`, …) würden sonst Fault 0
artifiziell abtrennen (innerhalb eines Splits) bzw. Train- und Test-Features
systematisch verschieden skalieren (zwischen den Splits: 480 vs. 800
Post-Fault-Samples). `run_length = 480` kürzt deshalb **jeden** Run auf die
ersten 480 Post-Fault-Samples = 24 h nach Fehlereintritt — in beiden Splits
dasselbe physikalische Fenster.

Dass die Projektionen daraus config-spezifisch 478–480 Samples machen, ist
davon unberührt — innerhalb jeder Konfiguration bleiben alle Runs und beide
Splits exakt gleich lang (siehe Kanallängen-Abschnitt oben).

## Projektionen: roh, DPCA, CVA, ICA

Alle Varianten bekommen **dieselbe Vorverarbeitung**, gesteuert über
`scaling_mode` — zwischen den Konfigurationen unterscheidet sich wirklich nur
die Projektion. Alle drei Verfahren werden **pro Run gefittet**, aus denselben
Gründen wie im Schwester-Notebook (Symmetrie der Methoden; DyCA/CVA suchen
ohnehin den Unterraum *dieses* Laufs).

### `scaling_mode` — entschärft gegenüber DyCA, aber nicht bedeutungslos

Die DC-Problematik des Schwester-Notebooks (DyCA-Amplituden bei
`"global_mean"` drei bis fünf Größenordnungen unter dem Gleichanteil) tritt
hier **nicht** auf: DPCA (sklearn-PCA), CVA (explizite Zentrierung der
Vergangenheits-/Zukunftsvektoren) und FastICA (interne Zentrierung) arbeiten
alle spaltenweise zentriert. Was bleibt, ist der PCA-Effekt: ohne Skalierung
dominieren die varianzstärksten Rohvariablen die Projektionsrichtungen (bei
CVA sind sogar die *kanonischen Korrelationen* skalierungsinvariant; die
extrahierten Variaten hängen über die Numerik der Ridge-Regularisierung
trotzdem minimal an der Skalierung). **Aktuell ist `"scaler"` eingestellt**
— konsistent mit der Umstellung aller Eigenwert-Notebooks auf
`scaler.transform(X)`; der Cache-Ordner (`tsfresh_cache_scaler`) ist vom
alten `global_mean`-Cache getrennt.

### Projektions-Definitionen

- **`dpca_n`**: Lag-Stacking `z_t = [x_t, x_{t−1}, …, x_{t−L}]` (L = 2,
  156 Spalten), dann PCA pro Run, Scores der ersten *n* Komponenten.
  Serienlänge 480 − L = **478**.
- **`cva_n`**: kanonische Variaten der Vergangenheit,
  `z_t = J^T (x_{t−1} − x̄)` mit J aus der CVA zwischen x_{t−1} und x_t
  (p = f = 1, Ridge wie im CVA-Notebook). Die Variaten haben per
  Konstruktion Einheitsvarianz. Serienlänge **479**.
- **`ica_n`**: FastICA pro Run mit *n* Quellen (`whiten="unit-variance"`,
  fester `random_state`). **Ordnungskonvention:** ICA-Komponenten sind
  unsortiert — die Kanäle werden nach |Exzess-Kurtosis| absteigend
  geordnet (stärkste Nicht-Gaussianität = `ic1`), dieselbe Konvention wie
  im `ICA_Eigenwerte.ipynb`. Ohne diese Ordnung wäre „Kanal ic1" über die
  Runs hinweg eine willkürlich gewürfelte Quelle und die TSFresh-Features
  wären Rauschen. Nicht-Konvergenz von FastICA ist kein Ausfall (die
  teil-konvergierte Rotation wird verwendet); nur echte Exceptions
  überspringen den Run. Serienlänge **480**.

**Vorzeichenkonvention.** Pro Run gefittete Achsen sind nur bis aufs
Vorzeichen bestimmt — bei allen drei Verfahren. TSFresh-Features wie `mean`,
`skewness` oder die Steigung von `linear_trend` kippen mit. `fix_signs=True`
dreht deshalb jeden projizierten Kanal so, dass sein betragsmäßig größter
Wert positiv ist — deterministisch und für alle Verfahren identisch (die
Kurtosis-Ordnung der ICA ist davon unberührt, das 4. Moment ist
vorzeicheninvariant).

Bleibt die Einschränkung, dass pro Run gefittete Achsen in verschiedenen Runs
verschiedene physikalische Richtungen sind. Formstatistiken (Autokorrelation,
Spektrum, Entropie) bleiben vergleichbar; lageabhängige Features sind mit
Vorsicht zu lesen. Das ist derselbe Preis wie im Schwester-Notebook.

In [ ]:
# ============================================================
# Pipeline anlegen
# ============================================================
# Prueft die Specs (Rangbedingungen der Verfahren, doppelte Namen) und
# fittet bei scaling_mode="scaler" den StandardScaler auf dem
# Normalbetrieb. Danach steht der Umfang des Laufs im Klartext da.
pipe = Pipeline(CFG)
pipe.describe()

## Phase A — Extraktion und Selektion auf dem Trainingssatz

Identisch zum Schwester-Notebook. Je Konfiguration:

1. **Extrahieren** in Chunks à `chunk_runs` Runs. Jeder Chunk wird als
   `float32`-Pickle gecacht — ein Abbruch kostet höchstens den angefangenen
   Chunk. Für `raw` liegen die Chunks (und die Top-`top_k`-Auswahl) bei
   bereits gelaufenem `TSFresh_PCA_DyCA.ipynb` schon im geteilten Cache.
2. **Selektieren:** Relevanztabelle in Spaltenblöcken (`block_cols`), damit
   die Roh-Konfiguration mit 40 404 Features nicht den Speicher sprengt.
3. **Reduzieren** auf `top_k` und die volle Matrix sofort freigeben.

**Zur Rangfolge.** Bei 10 500 Trainingsruns unterlaufen die p-Werte der
Signifikanztests reihenweise auf exakt 0.0 — nach p-Wert allein wären
hunderte Features gleichauf. Deshalb: `n_significant` (Zahl der Klassen, die
ein Feature signifikant trennt) absteigend, bei Gleichstand der
**ANOVA-F-Wert** — eine stetige Effektstärke, die nicht unterläuft.

Die Blockweise verschiebt die Benjamini-Hochberg-Korrektur minimal (sie sieht
je Block nur dessen p-Werte). Für eine *Rangfolge* ist das ohne Belang.

In [ ]:
# ============================================================
# PHASE A: Train extrahieren -> selektieren -> auf top_k reduzieren
# ============================================================
# Laeuft aus dem Chunk-Cache weiter, wenn schon Chunks vorhanden sind.
# Ergebnis liegt danach in pipe.train_top und pipe.top_names.
pipe.run_phase_a()

## Phase B — dieselben Features auf dem Testset

`from_columns()` übersetzt die ausgewählten Featurenamen zurück in ein
`kind_to_fc_parameters`-Dictionary. `extract_features` berechnet damit **nur**
diese Features — und auch nur auf den Kanälen, die in der Auswahl überhaupt
vorkommen. Deshalb ist Phase B um Größenordnungen billiger als Phase A. Die
FastICA-Fits der ica-Konfigurationen fallen allerdings erneut an (die
Projektion selbst lässt sich nicht aus Featurenamen sparen).

Die Selektion hat ausschließlich Trainingsdaten gesehen — das Testset bleibt
eine unverzerrte Generalisierungsschätzung.

In [ ]:
# ============================================================
# PHASE B: Testset - nur die in Phase A ausgewaehlten Features
# ============================================================
# Ergebnis liegt danach in pipe.test_top.
pipe.run_phase_b()

## Phase C — LazyClassifier je Konfiguration

Alle 16 Konfigurationen laufen mit demselben Klassifizierer-Satz. Zwei Dinge
sind bewusst so gesetzt (Begründungen wie im Schwester-Notebook):

**Identische Run-Menge.** CVA/ICA können an einzelnen Runs numerisch
scheitern und verlieren dann genau diese Runs. `restrict_to_common_runs=True`
schneidet deshalb alle Konfigurationen auf die Runs zu, die in **allen**
vorhanden sind — sonst würden die Setups auf unterschiedlichen (potenziell
unterschiedlich schweren) Testmengen bewertet.

**Macro-F1 als Kernzahl.** Balanced Accuracy ist der Macro-Recall und sieht
Precision nicht — sie belohnt Modelle, die eine Klasse als Sammelbecken
missbrauchen. Macro-F1 bestraft das. Beides wird berichtet; lazypredicts
eigene `F1 Score`-Spalte ist die *gewichtete* Variante und deshalb hier nicht
die Vergleichszahl.

**Modellauswahl über 5-fache CV — RandomForest bleibt der Hauptvergleich.**
Phase C läuft mit `lc_cv_folds = 5`: lazypredict kreuzvalidiert jedes Modell
zusätzlich auf den **Trainingsdaten** und liefert die Spalten `… CV Mean/Std`.
Die Auswahl „bestes Modell je Konfiguration" läuft darüber
(`BalancedAccCVMean`) und sieht das Testset nicht; berichtet werden weiterhin
die einmaligen Testwerte. Das alte Test-Maximum wird zum Vergleich mit
ausgegeben — es ist durch den Winner's Curse leicht optimistisch, und der
Abstand zwischen beiden zeigt, wie groß dieser Effekt hier ist.

Zwei Einschränkungen der CV:

- lazypredicts `F1 Score CV Mean` ist die **gewichtete** Variante; ein
  Macro-F1 aus der CV gibt es nicht. Selektionsmetrik ist deshalb
  `Balanced Accuracy CV Mean` — dieselbe Wahl wie `SELECT_METRIC` im
  Eigenwert-Notebook.
- Modelle **ohne `predict_proba`** (LinearSVC, Ridge, SGD, …) bekommen **keine**
  CV-Werte: lazypredict rechnet alle CV-Scorer gebündelt mit
  `error_score="raise"`, der ROC-AUC-Scorer braucht aber Wahrscheinlichkeiten —
  scheitert er, werden alle CV-Spalten dieses Modells geleert. Sie fallen damit
  aus der CV-Auswahl heraus (im Test-Maximum sind sie weiter dabei).

Der **RandomForest-Block** bleibt der belastbare Konfigurationsvergleich:
festes Modell, gar keine Auswahl.

> Kosten: die CV fittet jedes Modell fünfmal zusätzlich (Folds parallel,
> `n_jobs=-1`) — Phase C dauert grob das 2- bis 4-Fache. Die teuren Phasen A
> und B sind nicht betroffen und bleiben gecacht.

In [ ]:
# ============================================================
# PHASE C: LazyClassifier je Konfiguration
# ============================================================
# Schreibt die summary-CSV in den Cache-Ordner und legt sie zusaetzlich
# in pipe.summary ab. Die vollen Leaderboards stehen in
# pipe.leaderboards[name].
pipe.run_phase_c()

In [ ]:
# ============================================================
# Vergleich der Konfigurationen
# ============================================================
# Laeuft nach einem Kernel-Neustart auch OHNE Phase A/B/C: die summary
# liegt als CSV im Cache und wird von compare() nachgeladen. Vorher nur
# die Import-, Konfigurations- und Pipeline-Zelle ausfuehren.
cmp = pipe.compare()

In [ ]:
# ============================================================
# Plot: Hauptvergleich (RandomForest) + bestes Modell als Marker
# ============================================================
_ = pipe.plot_comparison(cmp)

## Confusion-Matrizen je Konfiguration

Dieselbe Auswertung wie am Ende von `LazyClassifier_PCA_DyCA.ipynb`, hier über
alle 16 Konfigurationen: **ein fester Modelltyp** — RandomForest, also der
Hauptvergleich von oben — wird pro Konfiguration auf dem **gesamten**
Trainingssatz gefittet und **einmal** auf dem echten Testset ausgewertet. Die
Unterschiede zwischen den Matrizen liegen damit allein an den Features.

**Warum neu gefittet wird.** Phase C berechnet die Vorhersagen zwar schon
(`predictions=True`), überschreibt `preds` aber je Konfiguration und hebt nichts
davon auf. Ein einzelner RF-Fit je Konfiguration auf `top_k = 100` Features
kostet Sekunden bis rund eine Minute — deutlich billiger, als Phase C mit ~25
Modellen zu wiederholen. Die Zelle gleicht am Ende gegen
`tsfresh_summary_dpca_cva_ica.csv` ab, ob sie die dortige RandomForest-Zeile
reproduziert.

**Datenbasis identisch zu Phase C:** gemeinsame Runs
(`restrict_to_common_runs`), dieselbe NaN/inf-Behandlung, `StandardScaler` +
RandomForest wie lazypredict intern. Die gemeinsamen Runs werden hier neu
bestimmt — die Zelle läuft also auch ohne vorher gelaufenes Phase C, solange
`pipe.train_top`/`pipe.test_top` aus Phase A/B im Kernel liegen.

> Der `StandardScaler` in dieser Pipeline ist **nicht** `scaling_mode` — er
> standardisiert die 100 TSFresh-Features vor dem Klassifizierer (so macht es
> lazypredict intern) und läuft in beiden Skalierungsmodi gleich.
> `scaling_mode = "scaler"` betrifft die Vorverarbeitung **vor** der Projektion,
> ist also längst in den Features eingebacken, die hier ankommen.

**Darstellung.** Die Matrizen sind **zeilenweise normiert** (Zeilensumme = 1):
Zelle (i, j) ist der Anteil der wahren Klasse *i*, der als *j* vorhergesagt
wurde, die Diagonale also der Recall. Alle Panels teilen sich die Skala 0…1 und
sind dadurch direkt vergleichbar. Die absoluten Zählwerte stehen als
21×21-DataFrame in `cm.counts[name]`.

> **Zum `raw`-Panel:** die Features sind bitidentisch zu denen im
> Schwester-Notebook, die *Runmenge* ist es nicht. `restrict_to_common_runs`
> schneidet dort auf die Runs zu, die auch DyCA überlebt haben (Test: 10446
> statt 10500) — hier scheitert keine Projektion, also bleiben alle 10500. Die
> `raw`-Matrizen beider Notebooks sind deshalb ähnlich, aber nicht identisch.

Die Vorhersagen werden als `tsfresh_cm_predictions_dpca_cva_ica.csv` im Cache
abgelegt (eigener Name, damit die Datei des Schwester-Notebooks im geteilten
`cache_dir` nicht überschrieben wird) — hier also in `tsfresh_cache_scaler/`,
getrennt vom `global_mean`-Lauf in `tsfresh_cache/`. Beide lassen sich damit
direkt gegeneinanderlegen (siehe letzte Zelle). Nach einem Kernel-Neustart
laufen die Plotzellen darunter ohne Phase A/B/C; `refit=True` erzwingt die
Neuberechnung.

In [ ]:
# ============================================================
# Confusion-Matrizen: RandomForest je Konfiguration
# ============================================================
# refit=False nutzt den Vorhersage-Cache im Cache-Ordner; nach einem
# Kernel-Neustart laufen die Plotzellen darunter damit ganz ohne
# Phase A/B/C. refit=True fittet neu - noetig, wenn ueber estimator=...
# ein anderes Modell verglichen werden soll.
cm = pipe.confusion(refit=False)

In [ ]:
# ============================================================
# Plot: alle Confusion-Matrizen im Raster
# ============================================================
_ = pipe.plot_confusions(cm, ncols=4)

In [ ]:
# ============================================================
# Detail: eine Konfiguration gross + groesste Verwechslungen
# ============================================================
# focus=None waehlt die beste Konfiguration nach Macro-F1; sonst z.B.
# focus="raw". annot_min ist die Schwelle, ab der eine Zelle beschriftet
# wird, top_n die Laenge der Verwechslungsliste.
_, focus = plot_confusion_detail(cm, focus=None, annot_min=0.05, top_n=8)

cm.counts[focus]

In [ ]:
# ============================================================
# Recall je Fault-Klasse und Konfiguration
# ============================================================
recall_tab = plot_recall(cm)

recall_tab.round(3)

## Wo weitergemacht werden kann

- **Feature-Namen ansehen:** `pipe.top_names["cva_8"]` zeigt, *welche*
  TSFresh-Features eine Konfiguration ausgewählt hat — oft aufschlussreicher
  als der Score. Die vollständige Rangtabelle liefert `rank_features`.
- **Quervergleich mit PCA/DyCA:** `tsfresh_summary.csv` (Schwester-Notebook)
  und `tsfresh_summary_dpca_cva_ica.csv` (dieses Notebook) liegen im selben
  Cache-Ordner und teilen die `raw`-Zeilen als gemeinsamen Anker — beide CSVs
  laden, konkatenieren und über alle 21 Konfigurationen plotten.
- **`top_k` variieren:** die Auswahl ist gecacht, ein anderer Wert erzwingt
  eine neue Selektion; die teure Extraktion in `cache_dir` bleibt gültig.
- **CV wieder abschalten:** `lc_cv_folds = 0` in der Konfigurationszelle nimmt
  die 5-fache Train-CV aus Phase C heraus und spart grob das 2- bis 4-Fache der
  Phase-C-Zeit; die Modellauswahl fällt dann auf das Test-Maximum zurück. Die
  teuren Phasen A/B sind nicht betroffen und bleiben gecacht.
- **Projektions-Parameter variieren:** `dpca_lags`, `cva_past`/`cva_fut`
  oder die FastICA-Einstellungen stehen in der Konfigurationszelle.
  ACHTUNG: die Chunk-Caches der betroffenen Konfigurationen (`dpca_*__...`,
  `cva_*__...`, `ica_*__...`) vorher löschen — der Cache-Name kodiert diese
  Parameter nicht, alte Chunks würden sonst stumm weiterverwendet.
- **Skalierung umstellen:** `scaling_mode = "scaler"` in der
  Konfigurationszelle; der Cache-Ordner bekommt den Modus automatisch
  angehängt, alte Ergebnisse werden nicht überschrieben.
- **Confusion-Matrizen:** das `cm`-Objekt aus der Zelle oben hält
  `cm.results` (Matrizen + Scores), `cm.counts` (absolute Zählwerte je
  Konfiguration) und `cm.recall_table()` (Recall je Klasse) bereit. Für
  ein anderes Modell `pipe.confusion(estimator=..., refit=True)` aufrufen
  — ohne `refit=True` wird der Vorhersage-Cache `tsfresh_cm_predictions_dpca_cva_ica.csv`
  weiterverwendet.
- **Skalierungsmodi vergleichen:** die Vorhersagen beider Läufe liegen als
  `tsfresh_cache/tsfresh_cm_predictions_dpca_cva_ica.csv` (`global_mean`) und
  `tsfresh_cache_scaler/tsfresh_cm_predictions_dpca_cva_ica.csv` (`scaler`) nebeneinander — beide laden,
  `cm.recall_table()` je Modus bauen und die Differenz ansehen zeigt, welche
  Fault-Klassen die Standardisierung tatsächlich rettet.

- **Eigenes Verfahren ergänzen:** ein neuer `Projector` wird über
  `tep.tsfresh.register("mein_verfahren", Projector(...))` eingetragen und
  ist danach als Spec `("mein_verfahren", …)` in `CFG.configs` nutzbar —
  an der Pipeline selbst muss dafür nichts geändert werden. Vorbild sind
  die sieben Einträge in `tep/tsfresh/projections.py`; ein `Projector`
  braucht nur `name`, `channels` und `apply`.